# UCS420: Cognitive Computing — Assignment 10
## Heart Health Information Assistant (Even Roll Number)

A Python-based **Healthcare Cognitive Assistant** that understands user queries about heart health, identifies their intent, reasons over a small domain ontology and a rule base, retrieves information from a knowledge base and generates an appropriate response.

**Pipeline**

`User query → Rules (safety checks) → NLP preprocessing → TF-IDF → Intent classifier → Confidence check → Knowledge retrieval + Concept reasoning → Response`

| # | Requirement | Section |
|---|---|---|
| 1 | Labelled corpus (≥40 queries, ≥5 intents) | §1 |
| 2 | Hierarchical taxonomy (≥4 categories × ≥2 subcategories) | §2 |
| 3 | ≥8 domain concepts, ≥10 relationships | §3 |
| 4 | Lowercasing, tokenization, stopword removal, lemmatization | §4 |
| 5 | TF-IDF + Naive Bayes / Logistic Regression / SVM | §5 |
| 6 | Accuracy, precision, recall, F1, confusion matrix | §6 |
| 7 | Domain knowledge base per intent | §7 |
| 8 | ≥5 rules incl. emergency, low-confidence, unsupported | §8 |
| 9 | Integrated assistant | §9 |
| 10 | Interactive Colab interface | §10 |

*Run all cells in order (Runtime → Run all).*

## 0. Setup

In [ ]:
# NLTK, scikit-learn, pandas, matplotlib, seaborn and networkx are preinstalled in Google Colab
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix)

print("Setup complete.")

## 1. Labelled Query Corpus
**120 queries across 6 intent categories** (20 per intent). Emergencies are deliberately *not* an intent class: they are caught by a rule (§8) **before** classification, so a misclassification can never hide an emergency.

In [ ]:
corpus = {
    "symptoms": [
        "What are the common symptoms of heart disease?",
        "What does a heart attack feel like?",
        "What are the warning signs of heart failure?",
        "Is shortness of breath a sign of a heart problem?",
        "Why do my legs and ankles swell, could it be my heart?",
        "What are the symptoms of an irregular heartbeat?",
        "How do women experience heart attack symptoms differently?",
        "Can heart palpitations be a sign of heart disease?",
        "What does angina chest discomfort feel like?",
        "Is feeling tired all the time a symptom of a weak heart?",
        "What are the early signs of blocked arteries?",
        "Can dizziness be caused by heart problems?",
        "My chest feels tight when I climb stairs, is that a heart symptom?",
        "Why do I get breathless when lying down at night?",
        "What are the signs of a weak heart?",
        "Can jaw or arm pain be a heart symptom?",
        "My heart is racing and fluttering, what does that mean?",
        "What symptoms show that my arteries are blocked?",
        "Do heart problems cause swelling in the feet?",
        "What are the symptoms of a silent heart attack?",
    ],
    "risk_factors": [
        "What are the main risk factors for heart disease?",
        "Does smoking increase the risk of a heart attack?",
        "How does high blood pressure damage the heart?",
        "Is heart disease hereditary or genetic?",
        "Can diabetes lead to heart problems?",
        "How does high cholesterol affect my heart?",
        "Does obesity raise the chance of cardiovascular disease?",
        "Does stress cause heart disease?",
        "At what age does the risk of heart disease increase?",
        "Does drinking alcohol harm the heart?",
        "Is a family history of heart attacks a risk for me?",
        "Can lack of sleep increase heart disease risk?",
        "Who is most at risk of getting heart disease?",
        "Does being overweight affect the heart?",
        "Can high blood sugar damage my heart?",
        "Are men more at risk of heart attack than women?",
        "Does vaping or chewing tobacco harm the heart?",
        "Can air pollution increase heart disease risk?",
        "Why does high LDL cholesterol increase heart attack risk?",
        "Does a sedentary lifestyle raise the risk of heart disease?",
    ],
    "diet_nutrition": [
        "What foods are good for heart health?",
        "Which foods should I avoid to keep my heart healthy?",
        "How much salt should I eat per day for a healthy heart?",
        "Is the DASH diet good for lowering blood pressure?",
        "Are eggs bad for cholesterol?",
        "What is a heart healthy breakfast?",
        "Does eating oily fish help the heart?",
        "Which cooking oil is best for the heart?",
        "Should I cut down on sugar for my heart?",
        "Are nuts good for lowering cholesterol?",
        "Is fried food and fast food bad for the heart?",
        "Can a vegetarian diet reduce heart disease?",
        "What should I eat to lower my blood pressure?",
        "Is coffee bad for the heart?",
        "Is ghee healthy for the heart?",
        "Which fruits and vegetables are best for the heart?",
        "How much fibre should I eat for heart health?",
        "Is red meat bad for my heart?",
        "Are packaged snacks and pickles bad for blood pressure?",
        "What drinks are good for heart health?",
    ],
    "exercise_lifestyle": [
        "How much exercise do I need for a healthy heart?",
        "What type of workout is best for heart health?",
        "Is walking daily good for the heart?",
        "How can I quit smoking to protect my heart?",
        "Can yoga and meditation improve heart health?",
        "How many hours of sleep are good for my heart?",
        "Is it safe to exercise after a heart attack?",
        "How can I manage stress to protect my heart?",
        "How can I lose weight to keep my heart healthy?",
        "Does sitting for long hours harm the heart?",
        "Is running or cycling better for cardio fitness?",
        "How do I start exercising if I have high blood pressure?",
        "How many steps a day should I walk for heart health?",
        "Is weight training safe for the heart?",
        "What breathing exercises help me relax and lower stress?",
        "How can I stay active if I have a desk job?",
        "Is swimming good exercise for the heart?",
        "How many days a week should I work out?",
        "Does quitting smoking really help the heart?",
        "How do I build a daily routine to keep my heart healthy?",
    ],
    "medication": [
        "What do statins do?",
        "Why do doctors prescribe aspirin for heart patients?",
        "What are the side effects of beta blockers?",
        "How do blood thinners work?",
        "What medicines are used to treat high blood pressure?",
        "What is nitroglycerin used for?",
        "Can ACE inhibitors cause a cough?",
        "Do I need to take heart medicine for life?",
        "What are diuretics or water pills used for?",
        "Can I take painkillers with my heart medication?",
        "What drugs are used to treat heart failure?",
        "What happens if I miss a dose of my blood pressure tablet?",
        "What is atorvastatin used for?",
        "Are there side effects of taking statins?",
        "What does clopidogrel do?",
        "Why was I prescribed a beta blocker?",
        "Is it safe to take aspirin every day?",
        "What are calcium channel blockers?",
        "What is warfarin used for?",
        "Do heart medicines interact with other drugs?",
    ],
    "diagnostic_tests": [
        "What is an ECG test?",
        "How is heart disease diagnosed?",
        "What does an echocardiogram show?",
        "What is a stress test for the heart?",
        "What is a normal blood pressure reading?",
        "What is a good cholesterol level?",
        "What is an angiography?",
        "How often should I check my blood pressure?",
        "What is a lipid profile test?",
        "What does a troponin blood test measure?",
        "What is a Holter monitor?",
        "What is a normal resting heart rate?",
        "What does an ECG show about my heart?",
        "Which blood tests check heart health?",
        "What is a TMT treadmill test?",
        "What is a CT coronary angiogram?",
        "How do I measure my blood pressure at home?",
        "What is a healthy HDL level?",
        "What does ejection fraction mean in an echo report?",
        "When should I get a heart checkup?",
    ],
}

df = pd.DataFrame([(q, intent) for intent, qs in corpus.items() for q in qs],
                  columns=["query", "intent"])
print(f"Total queries: {len(df)} | Intent categories: {df['intent'].nunique()}\n")
print(df["intent"].value_counts().to_string())
df.sample(8, random_state=1)

## 2. Hierarchical Taxonomy
**6 major categories**, each with **2–4 subcategories** (and example leaf topics). Each intent class maps onto a branch of this taxonomy.

In [ ]:
taxonomy = {
    "Heart Health": {
        "Cardiovascular Conditions": {
            "Coronary Artery Disease": ["Angina", "Heart Attack (Myocardial Infarction)"],
            "Heart Failure": ["Left-sided", "Right-sided"],
            "Arrhythmia": ["Atrial Fibrillation", "Bradycardia / Tachycardia"],
            "Hypertension": ["Primary", "Secondary"],
        },
        "Symptoms & Warning Signs": {
            "Common Symptoms": ["Fatigue", "Palpitations", "Swelling (Oedema)", "Dizziness"],
            "Emergency Signs": ["Severe Chest Pain", "Breathlessness at Rest", "Fainting"],
        },
        "Risk Factors": {
            "Modifiable": ["Smoking", "High Blood Pressure", "High Cholesterol", "Diabetes",
                           "Obesity", "Physical Inactivity", "Stress", "Alcohol"],
            "Non-Modifiable": ["Age", "Sex", "Family History / Genetics"],
        },
        "Prevention & Lifestyle": {
            "Diet & Nutrition": ["Low Salt", "Healthy Fats", "Fibre", "Limit Sugar"],
            "Physical Activity": ["Aerobic Exercise", "Strength Training", "Yoga"],
            "Stress, Sleep & Habits": ["Sleep Hygiene", "Stress Management", "Quitting Smoking"],
        },
        "Treatment": {
            "Medications": ["Statins", "Antiplatelets (Aspirin)", "Beta Blockers",
                            "ACE Inhibitors", "Diuretics", "Anticoagulants", "Nitrates"],
            "Procedures": ["Angioplasty & Stenting", "Bypass Surgery (CABG)", "Pacemaker"],
        },
        "Diagnosis & Monitoring": {
            "Clinical Tests": ["ECG", "Echocardiogram", "Stress Test (TMT)", "Angiography",
                               "Troponin", "Holter Monitor"],
            "Routine Monitoring": ["Blood Pressure", "Lipid Profile", "Resting Heart Rate"],
        },
    }
}

# Intent → taxonomy path (used by the assistant to show where an answer comes from)
intent_to_taxonomy = {
    "symptoms":           "Heart Health › Symptoms & Warning Signs",
    "risk_factors":       "Heart Health › Risk Factors",
    "diet_nutrition":     "Heart Health › Prevention & Lifestyle › Diet & Nutrition",
    "exercise_lifestyle": "Heart Health › Prevention & Lifestyle › Physical Activity / Stress, Sleep & Habits",
    "medication":         "Heart Health › Treatment › Medications",
    "diagnostic_tests":   "Heart Health › Diagnosis & Monitoring",
}

def print_tree(node, prefix=""):
    items = list(node.items()) if isinstance(node, dict) else [(leaf, None) for leaf in node]
    for i, (name, child) in enumerate(items):
        last = i == len(items) - 1
        print(prefix + ("└── " if last else "├── ") + name)
        if child:
            print_tree(child, prefix + ("    " if last else "│   "))

print("Heart Health")
print_tree(taxonomy["Heart Health"])

## 3. Domain Concepts & Relationships (Mini-Ontology)
**16 concepts** (each with a type, definition and surface forms used to spot it in a query) and **18 relationships** stored as `(subject, relation, object)` triples. The assistant uses this graph to add *related facts* to its answers.

In [ ]:
concepts = {
    "Heart":                   {"type": "Organ",      "synonyms": ["heart", "cardiac", "cardio"],
                                "definition": "Muscular organ that pumps blood through the body."},
    "Atherosclerosis":         {"type": "Process",    "synonyms": ["atherosclerosis", "plaque", "blocked arter", "clogged arter", "blockage"],
                                "definition": "Build-up of fatty plaque that narrows and hardens arteries."},
    "Coronary Artery Disease": {"type": "Condition",  "synonyms": ["coronary", "cad", "artery disease"],
                                "definition": "Narrowing of the arteries that supply the heart muscle."},
    "Heart Attack":            {"type": "Condition",  "synonyms": ["heart attack", "myocardial infarction", "mi "],
                                "definition": "Death of heart muscle when its blood supply is blocked."},
    "Heart Failure":           {"type": "Condition",  "synonyms": ["heart failure", "weak heart"],
                                "definition": "The heart cannot pump enough blood to meet the body's needs."},
    "Arrhythmia":              {"type": "Condition",  "synonyms": ["arrhythmia", "irregular heartbeat", "palpitation", "atrial fibrillation", "afib"],
                                "definition": "An abnormal heart rhythm — too fast, too slow or irregular."},
    "Hypertension":            {"type": "RiskFactor", "synonyms": ["hypertension", "high blood pressure", "blood pressure", " bp"],
                                "definition": "Persistently raised pressure of blood against artery walls."},
    "High Cholesterol":        {"type": "RiskFactor", "synonyms": ["cholesterol", "ldl", "triglyceride", "lipid"],
                                "definition": "Excess LDL cholesterol in the blood, which feeds plaque formation."},
    "Diabetes":                {"type": "RiskFactor", "synonyms": ["diabetes", "diabetic", "blood sugar"],
                                "definition": "Raised blood glucose, which damages blood vessels over time."},
    "Smoking":                 {"type": "RiskFactor", "synonyms": ["smok", "cigarette", "tobacco", "vaping"],
                                "definition": "Tobacco use; damages artery linings and promotes clotting."},
    "Obesity":                 {"type": "RiskFactor", "synonyms": ["obesity", "obese", "overweight", "weight"],
                                "definition": "Excess body fat that raises blood pressure, cholesterol and sugar."},
    "Chest Pain":              {"type": "Symptom",    "synonyms": ["chest pain", "angina", "chest discomfort", "chest tightness", "chest pressure"],
                                "definition": "Pain or pressure in the chest, often due to reduced blood flow to the heart."},
    "Physical Activity":       {"type": "Lifestyle",  "synonyms": ["exercise", "workout", "walking", "running", "cycling", "yoga", "physical activity"],
                                "definition": "Regular movement that strengthens the heart and improves risk factors."},
    "Healthy Diet":            {"type": "Lifestyle",  "synonyms": ["diet", "food", "salt", "fish", "nuts", "oil", "fruit", "vegetable", "eat"],
                                "definition": "Eating pattern low in salt, sugar and saturated fat, rich in plants."},
    "Statins":                 {"type": "Medication", "synonyms": ["statin", "atorvastatin", "rosuvastatin"],
                                "definition": "Drugs that lower LDL cholesterol by reducing its production in the liver."},
    "ECG":                     {"type": "Test",       "synonyms": ["ecg", "ekg", "electrocardiogram"],
                                "definition": "Records the electrical activity of the heart."},
}

relationships = [
    ("Smoking",                 "increases_risk_of", "Atherosclerosis"),
    ("High Cholesterol",        "causes",            "Atherosclerosis"),
    ("Atherosclerosis",         "leads_to",          "Coronary Artery Disease"),
    ("Coronary Artery Disease", "can_cause",         "Heart Attack"),
    ("Coronary Artery Disease", "has_symptom",       "Chest Pain"),
    ("Heart Attack",            "has_symptom",       "Chest Pain"),
    ("Heart Attack",            "damages",           "Heart"),
    ("Hypertension",            "increases_risk_of", "Heart Failure"),
    ("Hypertension",            "increases_risk_of", "Heart Attack"),
    ("Diabetes",                "increases_risk_of", "Coronary Artery Disease"),
    ("Obesity",                 "contributes_to",    "Hypertension"),
    ("Obesity",                 "contributes_to",    "Diabetes"),
    ("Physical Activity",       "reduces",           "Hypertension"),
    ("Physical Activity",       "reduces",           "Obesity"),
    ("Healthy Diet",            "reduces",           "High Cholesterol"),
    ("Statins",                 "treats",            "High Cholesterol"),
    ("ECG",                     "diagnoses",         "Arrhythmia"),
    ("ECG",                     "diagnoses",         "Heart Attack"),
]

print(f"Concepts: {len(concepts)} | Relationships: {len(relationships)}\n")
print(pd.DataFrame(relationships, columns=["Subject", "Relation", "Object"]).to_string(index=False))

In [ ]:
# Visualise the concept graph
G = nx.DiGraph()
for c, info in concepts.items():
    G.add_node(c, ctype=info["type"])
for s, r, o in relationships:
    G.add_edge(s, o, label=r)

palette = {"Organ": "#e76f51", "Process": "#f4a261", "Condition": "#e63946", "RiskFactor": "#e9c46a",
           "Symptom": "#ff8fab", "Lifestyle": "#2a9d8f", "Medication": "#457b9d", "Test": "#8d99ae"}
plt.figure(figsize=(15, 10))
pos = nx.spring_layout(G, k=1.6, seed=7)
nx.draw_networkx_nodes(G, pos, node_size=2600, node_color=[palette[G.nodes[n]["ctype"]] for n in G])
nx.draw_networkx_labels(G, pos, font_size=8, font_weight="bold")
nx.draw_networkx_edges(G, pos, arrows=True, arrowsize=18, edge_color="grey", node_size=2600)
nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, "label"), font_size=7)
for t, col in palette.items():
    plt.scatter([], [], c=col, s=120, label=t)
plt.legend(loc="lower left", fontsize=8)
plt.title("Heart Health Concept Graph")
plt.axis("off")
plt.show()

## 4. NLP Preprocessing
Lowercase conversion → tokenization (NLTK `word_tokenize`) → removal of punctuation & stopwords → lemmatization (WordNet, noun then verb form, e.g. *arteries → artery*, *eating → eat*).

In [ ]:
lemmatizer = WordNetLemmatizer()
STOPWORDS = set(stopwords.words("english"))

def preprocess(text, verbose=False):
    lowered = text.lower()                                              # 1. lowercase
    tokens = word_tokenize(lowered)                                     # 2. tokenize
    no_stop = [t for t in tokens if t.isalpha() and t not in STOPWORDS] # 3. stopword + punctuation removal
    lemmas = [lemmatizer.lemmatize(lemmatizer.lemmatize(t, pos="n"), pos="v") for t in no_stop]  # 4. lemmatize
    if verbose:
        print(f"Original     : {text}\nLowercased   : {lowered}\nTokens       : {tokens}\n"
              f"No stopwords : {no_stop}\nLemmatized   : {lemmas}\n")
    return " ".join(lemmas)

for q in ["Does smoking increase the risk of a heart attack?",
          "What are the early signs of blocked arteries?"]:
    preprocess(q, verbose=True)

df["clean"] = df["query"].apply(preprocess)
df[["query", "clean", "intent"]].head(10)

## 5. Feature Extraction (TF-IDF) & Classifier Training
TF-IDF over unigrams + bigrams, then three classifiers are compared on a stratified 75/25 split and with 5-fold cross-validation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], df["intent"], test_size=0.25, stratify=df["intent"], random_state=42)
print(f"Train: {len(X_train)} queries | Test: {len(X_test)} queries")

def make_tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)

models = {
    "Naive Bayes":         MultinomialNB(alpha=0.3),
    "Logistic Regression": LogisticRegression(C=10, max_iter=2000),
    "Linear SVM":          LinearSVC(C=1.0),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows, fitted = [], {}
for name, clf in models.items():
    pipe = Pipeline([("tfidf", make_tfidf()), ("clf", clf)]).fit(X_train, y_train)
    fitted[name] = pipe
    pred = pipe.predict(X_test)
    cv_acc = cross_val_score(Pipeline([("tfidf", make_tfidf()), ("clf", clf)]),
                             df["clean"], df["intent"], cv=cv, scoring="accuracy")
    rows.append({"Model": name,
                 "Test Accuracy": accuracy_score(y_test, pred),
                 "Precision (macro)": precision_score(y_test, pred, average="macro", zero_division=0),
                 "Recall (macro)": recall_score(y_test, pred, average="macro", zero_division=0),
                 "F1 (macro)": f1_score(y_test, pred, average="macro", zero_division=0),
                 "5-fold CV Accuracy": f"{cv_acc.mean():.3f} ± {cv_acc.std():.3f}"})

comparison = pd.DataFrame(rows).set_index("Model").round(3)
comparison

In [ ]:
vec = fitted["Logistic Regression"].named_steps["tfidf"]
print(f"TF-IDF vocabulary size: {len(vec.vocabulary_)}")
print("Sample features:", list(vec.get_feature_names_out()[:25]))

## 6. Evaluation of the Selected Model
**Logistic Regression** is chosen for the assistant: it performs on par with or better than the others here and, unlike LinearSVC, outputs **calibrated class probabilities**, which the low-confidence rule needs.

In [ ]:
best_name = "Logistic Regression"
best = fitted[best_name]
y_pred = best.predict(X_test)

print(f"=== {best_name} on held-out test set ===")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision : {precision_score(y_test, y_pred, average='macro', zero_division=0):.3f} (macro)")
print(f"Recall    : {recall_score(y_test, y_pred, average='macro', zero_division=0):.3f} (macro)")
print(f"F1-score  : {f1_score(y_test, y_pred, average='macro', zero_division=0):.3f} (macro)\n")
print(classification_report(y_test, y_pred, zero_division=0))

labels = sorted(df["intent"].unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Reds", xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted intent"); plt.ylabel("True intent")
plt.title(f"Confusion Matrix — {best_name}")
plt.xticks(rotation=35, ha="right"); plt.tight_layout(); plt.show()

In [ ]:
# Misclassified test queries (useful for the report)
errors = pd.DataFrame({"query": df.loc[X_test.index, "query"], "true": y_test, "predicted": y_pred})
errors = errors[errors["true"] != errors["predicted"]]
print(f"{len(errors)} misclassified test queries")
errors

In [ ]:
# Final model for deployment: retrain on the full corpus so the assistant uses every labelled example
final_model = Pipeline([("tfidf", make_tfidf()),
                        ("clf", LogisticRegression(C=10, max_iter=2000))]).fit(df["clean"], df["intent"])
print("Final model trained on all", len(df), "queries. Classes:", list(final_model.classes_))

## 7. Domain Knowledge Base
For each intent there is a **general** answer plus several **topic entries**. Retrieval picks the entry whose keywords best overlap the (lemmatised) query; if none match, the general answer is used.

*Content is general health information summarised from widely published guidance (WHO, American Heart Association, Indian clinical practice); it is not medical advice.*

In [ ]:
knowledge_base = {
    "symptoms": {
        "general": "Common signs of heart disease include chest pain or pressure (angina), shortness of breath, "
                   "palpitations, unusual tiredness, dizziness, and swelling of the legs, ankles or feet. "
                   "Symptoms that come on with exertion and ease with rest are especially worth reporting to a doctor.",
        "topics": [
            {"keywords": ["attack", "feel", "woman", "women", "differently"],
             "answer": "A heart attack typically causes pressure, squeezing or pain in the centre of the chest lasting "
                       "more than a few minutes, which may spread to the arm, jaw, neck or back, often with sweating, "
                       "nausea or breathlessness. Women, older adults and people with diabetes more often have "
                       "'atypical' symptoms such as breathlessness, nausea, back or jaw pain and extreme fatigue."},
            {"keywords": ["failure", "swell", "ankle", "leg", "tired", "weak", "fatigue", "breath", "shortness"],
             "answer": "Heart failure signs include breathlessness (especially when lying flat or on exertion), "
                       "swelling of the ankles and legs, rapid weight gain from fluid, persistent tiredness and a "
                       "reduced ability to exercise."},
            {"keywords": ["irregular", "heartbeat", "palpitation", "dizziness", "dizzy", "rhythm"],
             "answer": "Arrhythmias can feel like fluttering, racing, pounding or skipped beats, and may cause "
                       "dizziness, light-headedness or fainting. Occasional skipped beats are common, but frequent "
                       "or prolonged palpitations should be checked with an ECG."},
            {"keywords": ["angina", "chest", "discomfort", "block", "artery", "early"],
             "answer": "Angina is chest discomfort — tightness, heaviness or pressure — caused by reduced blood flow "
                       "through narrowed coronary arteries. It is usually triggered by exertion or stress and eases "
                       "with rest. It is often the earliest sign of blocked arteries."},
        ],
    },
    "risk_factors": {
        "general": "Major risk factors are high blood pressure, high LDL cholesterol, diabetes, smoking, obesity, "
                   "physical inactivity, unhealthy diet, excess alcohol and chronic stress (all modifiable), plus "
                   "age, male sex and family history (non-modifiable). Risk factors multiply each other's effect.",
        "topics": [
            {"keywords": ["smoke", "smoking", "tobacco", "cigarette", "vaping", "chew"],
             "answer": "Smoking (and other tobacco, including chewed tobacco and vaping) damages the lining of arteries, speeds up plaque build-up, raises blood pressure and "
                       "makes blood more likely to clot — roughly doubling heart-attack risk. Risk begins falling "
                       "within a year of quitting."},
            {"keywords": ["blood", "pressure", "hypertension"],
             "answer": "High blood pressure forces the heart to work harder and injures artery walls, leading to "
                       "thickened heart muscle, atherosclerosis, heart attack, heart failure and stroke. It often has "
                       "no symptoms, so regular checks matter."},
            {"keywords": ["cholesterol", "ldl", "lipid"],
             "answer": "Excess LDL ('bad') cholesterol deposits in artery walls as plaque, narrowing them and raising "
                       "the risk of heart attack. HDL ('good') cholesterol helps remove it."},
            {"keywords": ["diabetes", "sugar", "diabetic"],
             "answer": "Diabetes damages blood vessels and nerves over time; adults with diabetes are about two to "
                       "four times more likely to develop heart disease. Good sugar, BP and cholesterol control cuts this risk."},
            {"keywords": ["hereditary", "genetic", "family", "history", "age", "old"],
             "answer": "Family history matters, especially if a parent or sibling had heart disease early (before 55 in "
                       "men, 65 in women). Risk also rises with age — after about 45 in men and 55 in women. These can't "
                       "be changed, which makes controlling modifiable factors even more important."},
            {"keywords": ["obesity", "weight", "alcohol", "drink", "stress", "sleep"],
             "answer": "Obesity (especially abdominal fat) raises blood pressure, cholesterol and blood sugar. Heavy "
                       "drinking raises blood pressure and can weaken the heart muscle. Chronic stress and regularly "
                       "sleeping under 6–7 hours are also linked to higher blood pressure and heart risk."},
            {"keywords": ["men", "man", "woman", "sex", "pollution", "air", "sedentary", "inactive"],
             "answer": "Men tend to develop heart disease about 7–10 years earlier than women, though women's risk "
                       "rises sharply after menopause. Long-term exposure to air pollution (fine particles, PM2.5) and "
                       "a sedentary lifestyle are also recognised contributors to heart disease."},
        ],
    },
    "diet_nutrition": {
        "general": "A heart-healthy diet is rich in vegetables, fruit, whole grains, pulses, nuts and seeds, with fish "
                   "or lean protein, and limits salt, added sugar, refined carbs, saturated and trans fats and "
                   "processed or fried foods.",
        "topics": [
            {"keywords": ["salt", "sodium"],
             "answer": "WHO recommends less than 5 g of salt a day (about one teaspoon, ≈2,000 mg sodium). Most salt "
                       "comes from pickles, papads, namkeen, packaged and restaurant food rather than the salt shaker."},
            {"keywords": ["dash", "pressure", "blood"],
             "answer": "The DASH diet (Dietary Approaches to Stop Hypertension) emphasises vegetables, fruit, low-fat "
                       "dairy, whole grains, nuts and pulses while cutting salt and saturated fat. It can lower systolic "
                       "blood pressure by around 8–14 mmHg."},
            {"keywords": ["egg", "cholesterol", "nut", "lower"],
             "answer": "For most people, an egg a day doesn't significantly raise heart risk; saturated and trans fats "
                       "affect blood cholesterol more than dietary cholesterol. Nuts, oats, pulses and other soluble "
                       "fibre help lower LDL."},
            {"keywords": ["oil", "cook", "fry", "fried", "fast", "fish", "oily", "ghee", "butter"],
             "answer": "Prefer unsaturated oils (mustard, olive, groundnut, rice-bran, canola) in moderation and avoid "
                       "reused frying oil, vanaspati and trans fats. Ghee and butter are high in saturated fat, so use them sparingly. Fried and fast food are high in salt and unhealthy "
                       "fats. Oily fish (e.g. salmon, mackerel, sardines) provides omega-3s that support heart health."},
            {"keywords": ["sugar", "breakfast", "vegetarian", "plant"],
             "answer": "Cut added sugar and sugary drinks — they drive weight gain, diabetes and high triglycerides. A "
                       "heart-healthy breakfast could be oats or poha with vegetables, sprouts, fruit, or a besan "
                       "chilla. Well-planned vegetarian diets rich in pulses, nuts and whole grains are linked to "
                       "lower heart-disease risk."},
            {"keywords": ["coffee", "tea", "drink", "red", "meat", "fibre", "fiber", "fruit", "vegetable", "snack", "pickle", "packaged"],
             "answer": "Moderate coffee or tea (about 2–3 cups a day) is fine for most people; water, buttermilk and "
                       "unsweetened drinks are better than sugary or energy drinks. Limit red and processed meat. Aim for "
                       "at least 5 portions (≈400 g) of fruit and vegetables and about 25–30 g of fibre a day from whole "
                       "grains, pulses and vegetables. Packaged snacks and pickles are very high in salt."},
        ],
    },
    "exercise_lifestyle": {
        "general": "Aim for at least 150 minutes of moderate activity (e.g. brisk walking) or 75 minutes of vigorous "
                   "activity a week, plus muscle-strengthening on 2 days. Don't smoke, keep a healthy weight, sleep "
                   "7–9 hours and manage stress.",
        "topics": [
            {"keywords": ["walk", "run", "cycle", "type", "workout", "cardio", "fitness", "much", "exercise", "swim", "step", "week", "strength", "training"],
             "answer": "Aerobic exercise — brisk walking, cycling, swimming, running — is best for the heart: 30 "
                       "minutes on 5 days a week meets the 150-minute target. Add strength training twice a week. "
                       "Even a daily 30-minute walk measurably lowers blood pressure and risk; roughly 7,000–10,000 steps a day is a "
                       "useful everyday goal."},
            {"keywords": ["quit", "smoke", "smoking"],
             "answer": "To quit smoking: set a quit date, remove cigarettes and triggers, use nicotine replacement or "
                       "ask a doctor about medicines, and get support (India's National Tobacco Quitline is 1800-11-2356)."},
            {"keywords": ["yoga", "meditation", "stress", "manage", "relax"],
             "answer": "Yoga, meditation, deep breathing and regular exercise lower stress hormones and blood pressure. "
                       "Social connection, hobbies and good sleep also help; persistent stress or anxiety is worth "
                       "discussing with a professional."},
            {"keywords": ["sleep", "hour"],
             "answer": "Adults need 7–9 hours of good-quality sleep. Regular short sleep and untreated sleep apnoea "
                       "(loud snoring, pauses in breathing) raise blood pressure and heart risk."},
            {"keywords": ["attack", "safe", "after", "pressure", "start", "rehab"],
             "answer": "After a heart attack or with high blood pressure, exercise is usually encouraged but should be "
                       "cleared by your doctor first — ideally through a supervised cardiac-rehabilitation programme. "
                       "Start slowly (short walks) and build up gradually."},
            {"keywords": ["weight", "lose", "sit", "sedentary", "long", "desk", "job", "active", "routine"],
             "answer": "Long periods of sitting are linked to higher heart risk even in people who exercise — stand or "
                       "walk for a few minutes every 30–60 minutes. Losing even 5–10% of body weight improves blood "
                       "pressure, cholesterol and sugar."},
        ],
    },
    "medication": {
        "general": "Common heart medicines include statins (cholesterol), antiplatelets like aspirin and "
                   "anticoagulants (clot prevention), beta blockers, ACE inhibitors/ARBs, calcium-channel blockers "
                   "and diuretics (blood pressure/heart failure), and nitrates (angina). Always take them exactly "
                   "as prescribed.",
        "topics": [
            {"keywords": ["statin", "atorvastatin", "rosuvastatin", "muscle"],
             "answer": "Statins (e.g. atorvastatin, rosuvastatin) lower LDL cholesterol by reducing its production in "
                       "the liver and stabilise plaque, cutting heart-attack and stroke risk. Muscle aches are the most "
                       "reported side effect — tell your doctor rather than stopping on your own."},
            {"keywords": ["aspirin", "thinner", "clot", "anticoagulant", "clopidogrel", "warfarin", "apixaban"],
             "answer": "Aspirin and other antiplatelets stop platelets clumping; anticoagulants (e.g. warfarin, "
                       "apixaban) slow clot formation. They prevent heart attacks and strokes in people who have had "
                       "one or are high-risk, but raise bleeding risk — so they're not for everyone and should only be "
                       "taken on a doctor's advice."},
            {"keywords": ["beta", "blocker", "metoprolol", "prescribe"],
             "answer": "Beta blockers (e.g. metoprolol) slow the heart rate and lower blood pressure. Possible side "
                       "effects include tiredness, cold hands and feet, dizziness and a slow pulse."},
            {"keywords": ["ace", "inhibitor", "cough", "pressure", "calcium", "channel", "amlodipine"],
             "answer": "Blood pressure is treated with ACE inhibitors, ARBs, calcium-channel blockers, diuretics and "
                       "beta blockers, often in combination. A dry, persistent cough is a well-known ACE-inhibitor side "
                       "effect; doctors often switch to an ARB if it occurs. Calcium-channel blockers (e.g. amlodipine) relax "
                       "artery walls; ankle swelling is a common side effect."},
            {"keywords": ["nitroglycerin", "nitrate", "angina"],
             "answer": "Nitroglycerin (GTN, often a tablet under the tongue or a spray) widens blood vessels to relieve "
                       "angina chest pain quickly. Headache and light-headedness are common side effects."},
            {"keywords": ["diuretic", "water", "pill", "failure"],
             "answer": "Diuretics ('water pills', e.g. furosemide) help the kidneys remove extra salt and water, "
                       "reducing swelling and breathlessness in heart failure and lowering blood pressure. Heart failure "
                       "is also treated with ACE inhibitors/ARNIs, beta blockers, MRAs and SGLT2 inhibitors."},
            {"keywords": ["life", "miss", "dose", "painkiller", "forget", "interact", "drug"],
             "answer": "Many heart medicines are long-term; never stop them without your doctor's advice. If you miss "
                       "a dose, check the leaflet or ask your pharmacist — generally take it when you remember unless "
                       "the next dose is near, and don't double up. Avoid NSAID painkillers (e.g. ibuprofen) unless your "
                       "doctor approves, as they can raise blood pressure and interact with blood thinners."},
        ],
    },
    "diagnostic_tests": {
        "general": "Heart disease is diagnosed using your history and examination plus tests such as ECG, "
                   "echocardiogram, stress test (TMT), blood tests (lipid profile, troponin), Holter monitoring, "
                   "CT coronary angiography and invasive coronary angiography.",
        "topics": [
            {"keywords": ["ecg", "ekg", "electrocardiogram"],
             "answer": "An ECG records the heart's electrical activity using stickers on the chest. It's quick and "
                       "painless and detects arrhythmias, signs of a current or past heart attack and thickened heart muscle."},
            {"keywords": ["echocardiogram", "echo", "ultrasound", "ejection", "fraction"],
             "answer": "An echocardiogram is an ultrasound scan of the heart showing its chambers, valves and how well "
                       "it pumps (ejection fraction, normally about 55–70%)."},
            {"keywords": ["stress", "treadmill", "tmt", "exercise"],
             "answer": "A stress test (TMT) records your ECG while you walk on a treadmill, to reveal reduced blood flow "
                       "that only appears when the heart works harder."},
            {"keywords": ["pressure", "blood", "normal", "reading", "check", "often", "home", "measure", "checkup"],
             "answer": "Normal blood pressure is below 120/80 mmHg. Readings consistently at or above 130/80 (US "
                       "guidelines) or 140/90 (many other guidelines, including India's) indicate hypertension. Adults "
                       "should check at least once a year — more often if it is raised or you're on treatment. At home, use a validated "
                       "upper-arm monitor after 5 minutes' seated rest and take two readings a minute apart. A full heart "
                       "check-up (BP, lipids, sugar) is generally advised from about age 30–40, earlier with risk factors."},
            {"keywords": ["cholesterol", "lipid", "profile", "level", "good", "hdl", "ldl"],
             "answer": "A lipid profile is a blood test (often fasting) measuring total cholesterol, LDL, HDL and "
                       "triglycerides. Typical desirable values: total < 200 mg/dL, LDL < 100 mg/dL (lower for "
                       "high-risk people), HDL > 40 (men) / > 50 (women) mg/dL, triglycerides < 150 mg/dL."},
            {"keywords": ["angiography", "angiogram", "blockage", "ct", "coronary"],
             "answer": "Coronary angiography injects contrast dye into the heart's arteries and takes X-ray images to "
                       "show blockages; it can be followed directly by angioplasty and stenting if needed. A CT coronary angiogram is a "
                       "non-invasive version that uses a CT scan instead of a catheter."},
            {"keywords": ["troponin"],
             "answer": "Troponin is a protein released into the blood when heart muscle is damaged; a raised level is "
                       "the key blood test for diagnosing a heart attack."},
            {"keywords": ["holter", "monitor"],
             "answer": "A Holter monitor is a portable ECG worn for 24–48 hours (or longer) to catch intermittent "
                       "rhythm problems during everyday activity."},
            {"keywords": ["rate", "rest", "resting", "pulse"],
             "answer": "A normal resting heart rate for adults is 60–100 beats per minute; fit people may be lower. "
                       "Consistently above 100 or below 50 with symptoms should be checked."},
        ],
    },
}

for intent, kb in knowledge_base.items():
    print(f"{intent:<20} general answer + {len(kb['topics'])} topic entries")

In [ ]:
def retrieve(intent, clean_query):
    """Return the KB answer for an intent whose keywords best overlap the lemmatised query."""
    tokens = set(clean_query.split())
    best_entry, best_score = None, 0
    for entry in knowledge_base[intent]["topics"]:
        kw = {preprocess(k) for k in entry["keywords"]}
        score = len(tokens & kw)
        if score > best_score:
            best_entry, best_score = entry, score
    return best_entry["answer"] if best_entry else knowledge_base[intent]["general"]

print(retrieve("medication", preprocess("What are the side effects of statins?")))

## 8. Rule-Based Reasoning
Rules are checked in priority order. The first three are *safety* rules that run **before** the classifier.

| # | Rule | Condition | Action |
|---|---|---|---|
| R1 | Empty / greeting / help | blank input, greeting or "help" | Greet and show what the assistant can do |
| R2 | **Emergency escalation** | red-flag phrases (crushing chest pain, can't breathe, fainted, stroke signs…) | Stop and tell the user to call 112 / 108 now |
| R3 | **Unsupported query** | non-cardiac health topic, or no heart-health vocabulary at all | Explain scope, suggest an appropriate professional |
| R4 | Personal dosage / stop-medication | "how much X should I take", "can I stop my tablets" | No dosing advice → refer to doctor/pharmacist (general info still given if possible) |
| R5 | **Low confidence** | top intent probability < threshold | Ask to rephrase, show top-2 guesses |
| R6 | Multiple risk factors | ≥ 2 risk-factor concepts in one query | Add advice to get a cardiovascular risk assessment |
| R7 | Medical disclaimer | every informational answer | Append "not a substitute for a doctor" note |

In [ ]:
CONFIDENCE_THRESHOLD = 0.35  # tuned: vague queries score ~0.2–0.3, clear rewordings ≥0.45

GREETINGS = {"hi", "hello", "hey", "help", "namaste", "good morning", "good evening", "hii", "start"}

EMERGENCY_PATTERNS = [
    r"\b(severe|crushing|unbearable|intense|sudden|sharp)\b.*\bchest\b",
    r"\bchest (pain|pressure|tightness)\b.*\b(right now|now|minutes|won'?t go|not going away|sweat|spreading|radiat)",
    r"\b(pain|ache)\b.*\b(spread|spreading|radiating|going)\b.*\b(arm|jaw|neck|back|shoulder)",
    r"\b(having|i think i.?m having|is this) an? heart attack\b",
    r"\bhaving a heart attack\b",
    r"\b(can'?t|cannot|can not|unable to|struggling to|hard to) breathe\b",
    r"\b(fainted|passed out|unconscious|collapsed|not breathing|no pulse|unresponsive)\b",
    r"\bcardiac arrest\b",
    r"\b(face drooping|slurred speech|sudden numbness|sudden weakness)\b",
    r"\b(lips|face) (turning|are|is) blue\b",
]

NON_CARDIAC_TERMS = [
    "skin", "rash", "acne", "eczema", "hair", "tooth", "teeth", "dental", "eye", "vision", "ear",
    "pregnan", "period", "menstrua", "kidney stone", "cancer", "tumour", "tumor", "fever", "cold", "flu",
    "covid", "cough syrup", "migraine", "headache", "stomach", "acidity", "constipation", "diarrh",
    "fracture", "bone", "joint", "arthritis", "asthma", "thyroid", "depression", "anxiety disorder",
    "allergy", "liver", "insulin dose", "vaccine", "back pain", "knee",
]

CARDIAC_TERMS = ["heart", "cardi", "chest", "blood pressure", "bp", "cholesterol", "artery", "arteries",
                 "pulse", "palpitation", "angina", "statin", "ecg", "stroke", "hypertension", "lipid"]

DOSAGE_PATTERNS = [
    r"\bhow (much|many)\b.*\b(should|can|do) i (take|have)\b.*\b(mg|tablet|pill|aspirin|statin|medicine|medication|dose|drug)",
    r"\bhow (much|many)\b.*\b(mg|tablets?|pills?)\b",
    r"\b(what|which|right|correct) (dose|dosage)\b",
    r"\b(good|safe|right|correct|normal|usual|maximum|max|daily|ideal) (dose|dosage)\b",
    r"\bdosage\b",
    r"\bcan i (stop|quit|skip|double|increase|reduce)\b.*\b(medicine|medication|tablet|pill|dose|statin|aspirin|drug)",
    r"\b(stop|double|increase|reduce) (taking )?my\b.*\b(medicine|medication|tablet|pill|dose|statin|aspirin)",
]

RISK_CONCEPTS = {"Smoking", "Hypertension", "High Cholesterol", "Diabetes", "Obesity"}

# Domain vocabulary = every lemma seen in the corpus or knowledge base (used by R3)
DOMAIN_VOCAB = set(" ".join(df["clean"]).split())
for kb in knowledge_base.values():
    for entry in kb["topics"]:
        DOMAIN_VOCAB |= set(preprocess(" ".join(entry["keywords"])).split())
GENERIC = {"get", "know", "tell", "need", "good", "bad", "best", "help", "make", "keep", "take", "feel",
           "time", "day", "use", "way", "normal", "body", "health", "healthy", "much", "many", "long", "safe"}
HEART_VOCAB = DOMAIN_VOCAB - GENERIC

EMERGENCY_MSG = ("🚨 **This may be a medical emergency.** Call **112** (national emergency) or **108** "
                 "(ambulance) **right now**, or get someone to take you to the nearest emergency department. "
                 "Do not drive yourself. Stop any activity, sit or lie down, loosen tight clothing and stay on the "
                 "line with the operator. If the person is unresponsive and not breathing normally, start CPR "
                 "(hard, fast chest compressions in the centre of the chest).")

DISCLAIMER = "ℹ️ This is general information, not a diagnosis. Please consult a doctor for advice about your own health."


def contains_any(text, terms):
    return any(t in text for t in terms)

def find_concepts(text):
    t = " " + text.lower() + " "
    return [c for c, info in concepts.items() if any(s in t for s in info["synonyms"])]

def related_facts(found, limit=3):
    facts = []
    for s, r, o in relationships:
        if (s in found or o in found) and s != "Heart" and o != "Heart":
            facts.append(f"{s} —{r.replace('_', ' ')}→ {o}")
    return facts[:limit]

def rule_greeting(text):            # R1
    t = text.strip().lower().strip("!.? ")
    return t == "" or t in GREETINGS

def rule_emergency(text):           # R2
    t = text.lower()
    return any(re.search(p, t) for p in EMERGENCY_PATTERNS)

def rule_unsupported(text, clean):  # R3
    t = text.lower()
    if contains_any(t, NON_CARDIAC_TERMS) and not contains_any(t, CARDIAC_TERMS):
        return "non-cardiac"
    if not (set(clean.split()) & HEART_VOCAB):
        return "out-of-domain"
    return None

def rule_dosage(text):              # R4
    t = text.lower()
    return any(re.search(p, t) for p in DOSAGE_PATTERNS)

print("Rules defined. Domain vocabulary size:", len(HEART_VOCAB))

## 9. The Integrated Cognitive Assistant
`HeartHealthAssistant.respond(query)` chains everything: **rules → NLP → TF-IDF + Logistic Regression → confidence check → knowledge retrieval → concept-graph reasoning → response**.

In [ ]:
class HeartHealthAssistant:
    def __init__(self, model, threshold=CONFIDENCE_THRESHOLD):
        self.model = model
        self.threshold = threshold
        self.history = []

    def classify(self, clean):
        probs = self.model.predict_proba([clean])[0]
        order = np.argsort(probs)[::-1]
        return [(self.model.classes_[i], float(probs[i])) for i in order]

    def respond(self, query):
        result = {"query": query, "intent": None, "confidence": None, "rules": [],
                  "answer": "", "concepts": [], "facts": [], "taxonomy": None}

        # R1 — greeting / empty input
        if rule_greeting(query):
            result["rules"].append("R1 Greeting/Help")
            result["answer"] = ("👋 Hello! I'm a Heart Health Information Assistant. Ask me about heart "
                                "**symptoms**, **risk factors**, **diet**, **exercise & lifestyle**, "
                                "**medications** or **tests & normal values**.")
            return self._log(result)

        # R2 — emergency escalation (checked on the raw text, before anything else)
        if rule_emergency(query):
            result["rules"].append("R2 Emergency escalation")
            result["intent"] = "EMERGENCY"
            result["answer"] = EMERGENCY_MSG
            return self._log(result)

        clean = preprocess(query)
        result["clean"] = clean

        # R3 — unsupported query
        unsupported = rule_unsupported(query, clean)
        if unsupported:
            result["rules"].append(f"R3 Unsupported query ({unsupported})")
            result["intent"] = "UNSUPPORTED"
            result["answer"] = ("I can only help with **heart health** topics (symptoms, risk factors, diet, "
                                "exercise, heart medicines and heart tests). For this question, please consult "
                                "a doctor or the appropriate specialist.")
            return self._log(result)

        # Intent classification
        ranked = self.classify(clean)
        intent, conf = ranked[0]
        result["intent"], result["confidence"] = intent, round(conf, 3)
        result["top2"] = [(i, round(p, 3)) for i, p in ranked[:2]]

        # R4 — personal dosage / stopping medicine
        if rule_dosage(query):
            result["rules"].append("R4 Personal dosage request")
            result["intent"] = "medication"
            result["answer"] = ("💊 I can't recommend doses or tell you to start, stop or change a medicine — "
                                "that depends on your health, other medicines and test results. Please ask your "
                                "doctor or pharmacist. General information: "
                                + retrieve("medication", clean))
            result["answer"] += "\n\n" + DISCLAIMER
            return self._log(result)

        # R5 — low confidence
        if conf < self.threshold:
            result["rules"].append(f"R5 Low confidence ({conf:.2f} < {self.threshold})")
            g1, g2 = ranked[0][0], ranked[1][0]
            result["answer"] = (f"🤔 I'm not confident I understood. Did you mean something about "
                                f"**{g1.replace('_', ' ')}** or **{g2.replace('_', ' ')}**? "
                                "Please rephrase with a bit more detail, e.g. *'What are the symptoms of heart "
                                "failure?'* or *'Which foods lower cholesterol?'*")
            return self._log(result)

        # Knowledge retrieval + concept reasoning
        answer = retrieve(intent, clean)
        found = find_concepts(query)
        result["concepts"] = found
        result["facts"] = related_facts(found)
        result["taxonomy"] = intent_to_taxonomy[intent]

        # R6 — multiple risk factors
        risks = [c for c in found if c in RISK_CONCEPTS]
        if len(risks) >= 2:
            result["rules"].append("R6 Multiple risk factors")
            answer += (f"\n\n⚠️ You mentioned several risk factors ({', '.join(risks)}). Their effects add up, "
                       "so a full cardiovascular risk check (BP, lipid profile, blood sugar) with a doctor is a good idea.")

        # R7 — disclaimer
        result["rules"].append("R7 Disclaimer")
        result["answer"] = answer + "\n\n" + DISCLAIMER
        return self._log(result)

    def _log(self, result):
        self.history.append(result)
        return result

    @staticmethod
    def format(r):
        lines = [f"🧑 You: {r['query']}"]
        meta = f"   [intent: {r['intent']}"
        if r["confidence"] is not None:
            meta += f" | confidence: {r['confidence']:.2f}"
        meta += f" | rules: {', '.join(r['rules']) or '-'}]"
        lines.append(meta)
        if r.get("taxonomy"):
            lines.append(f"   [taxonomy: {r['taxonomy']}]")
        lines.append("🤖 Assistant: " + r["answer"].replace("**", "").replace("*", ""))
        if r["facts"]:
            lines.append("   🔗 Related: " + "; ".join(r["facts"]))
        return "\n".join(lines)


assistant = HeartHealthAssistant(final_model)
print(assistant.format(assistant.respond("What are the side effects of statins?")))

## Testing the Assistant
The five test types required by the assignment.

In [ ]:
test_cases = [
    # 1. One query per intent
    ("Intent: symptoms",            "What are the signs of a heart attack?"),
    ("Intent: risk_factors",        "Does smoking increase the risk of heart disease?"),
    ("Intent: diet_nutrition",      "Which foods help lower cholesterol?"),
    ("Intent: exercise_lifestyle",  "How much exercise should I do every week for my heart?"),
    ("Intent: medication",          "What are the side effects of statins?"),
    ("Intent: diagnostic_tests",    "What is a normal blood pressure reading?"),
    # 2. Differently worded queries
    ("Reworded (symptoms)",         "My feet and ankles are swollen and I get out of breath easily, could my heart be the cause?"),
    ("Reworded (diet)",             "Which cooking fat is kinder to my arteries, ghee or mustard oil?"),
    ("Reworded (exercise)",         "What workouts keep the heart fit?"),
    ("Reworded (tests)",            "Why would a cardiologist ask me to wear a Holter device?"),
    # 3. Unsupported healthcare queries
    ("Unsupported",                 "How do I treat a skin rash on my arm?"),
    ("Unsupported",                 "What is the best medicine for a migraine?"),
    # 4. Low-confidence query
    ("Low confidence",              "Tell me about my heart"),
    ("Low confidence",              "Which pills bring down cholesterol?"),
    # 5. Emergency queries
    ("Emergency",                   "I have crushing chest pain spreading to my left arm and I'm sweating"),
    ("Emergency",                   "My father just collapsed and he can't breathe"),
    # Extra rules
    ("Dosage rule",                 "How many mg of aspirin should I take daily?"),
    ("Multiple risk factors",       "I smoke and have diabetes, how does that affect my heart?"),
]

summary = []
for label, q in test_cases:
    r = assistant.respond(q)
    print(f"━━━━━━━━━━ {label} ━━━━━━━━━━")
    print(assistant.format(r), "\n")
    summary.append({"Test": label, "Query": q, "Intent": r["intent"],
                    "Confidence": r["confidence"], "Rules fired": ", ".join(r["rules"])})

In [ ]:
pd.set_option("display.max_colwidth", 70)
pd.DataFrame(summary)

## 10. Interactive Interface (Google Colab)
Type a question and press **Ask** (or Enter). Ask as many questions as you like; the conversation history stays on screen. **Clear** resets it.

In [ ]:
import html as _html
import ipywidgets as widgets
from IPython.display import display, HTML

try:  # enables third-party widget rendering in Colab (harmless elsewhere)
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

def _md(text):
    t = _html.escape(text)
    t = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", t)
    t = re.sub(r"\*(.+?)\*", r"<i>\1</i>", t)
    return t.replace("\n", "<br>")

def _bubble(r):
    colour = {"EMERGENCY": "#ffe5e5", "UNSUPPORTED": "#fff4e0"}.get(r["intent"], "#eef6ff")
    if any(x.startswith("R5") for x in r["rules"]):
        colour = "#fffbe0"
    meta = f"intent: <b>{r['intent']}</b>"
    if r["confidence"] is not None:
        meta += f" · confidence: {r['confidence']:.2f}"
    meta += f" · rules: {', '.join(r['rules']) or '—'}"
    facts = ("<div style='margin-top:6px;font-size:12px;color:#555'>🔗 " + " · ".join(map(_html.escape, r['facts'])) + "</div>") if r["facts"] else ""
    return (f"<div style='text-align:right;margin:8px 0'><span style='background:#d1f2d9;padding:8px 12px;"
            f"border-radius:12px;display:inline-block;max-width:75%'>🧑 {_html.escape(r['query'])}</span></div>"
            f"<div style='background:{colour};padding:10px 14px;border-radius:12px;margin:4px 0 12px;max-width:85%'>"
            f"🤖 {_md(r['answer'])}{facts}"
            f"<div style='margin-top:6px;font-size:11px;color:#777'>{meta}</div></div>")

chat_box = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="8px", height="420px", overflow="auto"))
query_in = widgets.Text(placeholder="Ask about heart health… e.g. 'What is a normal cholesterol level?'",
                        layout=widgets.Layout(width="70%"))
ask_btn = widgets.Button(description="Ask", button_style="danger", icon="heart")
clear_btn = widgets.Button(description="Clear", icon="trash")

def on_ask(_=None):
    q = query_in.value
    query_in.value = ""
    with chat_box:
        display(HTML(_bubble(assistant.respond(q))))

def on_clear(_=None):
    chat_box.clear_output()
    with chat_box:
        display(HTML("<i>❤️ Heart Health Assistant — ask me anything about heart health.</i>"))

ask_btn.on_click(on_ask)
clear_btn.on_click(on_clear)
try:
    query_in.on_submit(on_ask)          # Enter key (ipywidgets 7, as in Colab)
except Exception:
    query_in.continuous_update = False  # ipywidgets 8 fallback
    query_in.observe(lambda c: on_ask() if c["new"] else None, names="value")

on_clear()
display(widgets.HTML("<h3>❤️ Heart Health Information Assistant</h3>"),
        chat_box, widgets.HBox([query_in, ask_btn, clear_btn]))

### Alternative: text-based chat loop
If widgets don't render in your environment, run this cell instead. Type `quit` to stop.

In [ ]:
def chat_loop():
    print("❤️ Heart Health Assistant (type 'quit' to exit)\n")
    while True:
        q = input("You: ")
        if q.strip().lower() in {"quit", "exit", "bye"}:
            print("Assistant: Take care of your heart! Goodbye. 👋")
            break
        print(assistant.format(assistant.respond(q)).split("\n", 1)[1], "\n")

# Uncomment to run:
# chat_loop()

## Conclusion
- **Corpus & taxonomy:** 120 labelled queries over 6 intents, organised under a 6-branch heart-health taxonomy.
- **Ontology:** 16 concepts and 18 typed relationships, used at run time to attach related facts to answers.
- **NLP + ML:** lowercasing, tokenization, stopword removal and lemmatization feed a TF-IDF (uni+bigram) representation; Naive Bayes, Logistic Regression and Linear SVM were compared, and Logistic Regression was deployed for its probability outputs.
- **Reasoning:** 7 prioritised rules — safety rules (emergency, unsupported scope) run before the classifier so they can't be overridden by a misclassification; low-confidence and dosage rules prevent misleading answers.
- **Limitations:** a small hand-written corpus limits generalisation to unusual phrasing; keyword-based emergency detection can miss creatively worded emergencies (and deliberately errs on the side of over-triggering, e.g. any mention of *sudden chest pain*), so a real system would need a larger corpus, a dedicated emergency classifier and clinical review of the knowledge base.